# Volatility Forecasting — Research Session

You are a volatility trader. This afternoon you will quote two-sided markets on 
the (anualized) volatility over the next 20 trading days.

Your job in the next hour: **build a forecast you are willing to quote on.**

You have about 8 years of daily OHLC data on a single stock, with a column of
event labels. The data is not a real price history, but it inherits the 
statistical behaviour of one.

**Part 1: Modelling.** You will build *one* forecasting function, `forecast()`.
It is written for you and you never have to edit it. Instead, it reads its 
parameters  from a dictionary called `CONFIG`, and in every section of this 
notebook you decide what values these parameters attain. In the end, you will 
have optimized your forecasting function and and use it to predict the average 
annualized vol over the next 20 days.

**Part 2: Trading.** Once you have your forecast, you will trade with the other 
teams. You will trade volatility using contracts that carry a notional of 1000 
vega (= 1000 euros per volpoint). This is called a vol swap and if you buy one contract ("one lot")
at 20% volatility, while 22% actually realizes, you earn 
2000 euros. You will trade with eachother, and with a broker.

| section | what you decide | `CONFIG` key |
|---|---|---|
| 2 | how to measure the current vol level? | `lookback`, `lam`, `w` |
| 3 | does the *direction* of the recent move matter? | `lev_g`, `lev_n`|
| 4 | do scheduled events raise your quote? | `event_mult`, `exclude_events` |


**Before you touch anything: `File → Save a copy in Drive`.**
Otherwise your work disappears when you close the tab.

Below, instructions are color coded to indicate you simply need to <font color='greeb'>run</font> a cell, <font color='orange'>edit</font> a cell, or that you are encouraged to <font color='yellow'> discuss </font> the results of a cell with your team.


---
## 0 · Setup

<font color='greeb'>Run</font> the cell below to start loading data and check the print for a birds eye view of what we have at our hands. In the cell you will fnd a dictionary called *CONFIG* that we will tune in this notebook.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})
ANN = 252

CONFIG = {
    # --- section 2: the base level -----------------------------------
    "lookback":   None,    # days of history for the long-run average; None = all
    "lam":        0.94,    # EWMA decay for the recent-vol estimate
    "w":          0.0,     # weight on recent vol.  0 = ignore it entirely
    # --- section 3: leverage -----------------------------------------
    "lev_g":      0.0,     # 0 = direction does not change your quote at all
    "lev_n":      10,      # how many days back to measure the move
    # --- section 4: events -------------------------------------------
    "event_mult": {"earnings": 1.0, "agm": 1.0, "news": 1.0},   # 1.0 = no effect
    "exclude_events": [],      # event labels left OUT of the base level; [] = keep them all
    # --- fixed --------------------------------------------------------
    "horizon":    20,      # swap tenor, days
}

DATA_URL = "https://raw.githubusercontent.com/avspaendonck/quant/refs/heads/main/training_sample.csv"              # <-- instructor pastes the URL here

src = DATA_URL or "training_sample.csv"
try:
    df = pd.read_csv(src, parse_dates=["date"])
except Exception:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(next(iter(up)), parse_dates=["date"])

df["event_type"] = df["event_type"].fillna("")
df["comment"]    = df["comment"].fillna("")
df["ret"]        = np.log(df["close"]).diff()
df = df.reset_index(drop=True)

TEST_START = len(df) - 400     # held out. Nothing you tune may look past here.
SETTLE_COL = "v_c2c"           # the swap settles on close-to-close realised
                               # variance. This is the contract, not a choice.

def add_estimators(d):
    d["v_c2c"] = np.log(d["close"] / d["close"].shift(1))**2
    return d

df = add_estimators(df)

print(f"{len(df)} rows   {df.date.min():%Y-%m-%d} -> {df.date.max():%Y-%m-%d}")
print(f"unconditional annualised vol: {df.ret.std()*np.sqrt(ANN)*100:.1f}%")
print(f"training rows 0-{TEST_START-1}, held out {TEST_START}-{len(df)-1}")
print()
print("event labels present:")
print(df.event_type.value_counts().drop("", errors="ignore").to_string())
df.head()

---
## 1 · Explore the data

Let's explore the data. Below we plot stock price, the daily returns and volatility as computed over a 20 rolling (uniform) window. Simply **<font color='greeb'>run</font>** the cell.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(df.date, df.close, lw=.8, color="#1f3a63"); ax[0].set_ylabel("price")
ax[1].plot(df.date, df.ret*100, lw=.5, color="#555");  ax[1].set_ylabel("return %")
ax[2].plot(df.date, df.ret.rolling(20).std()*np.sqrt(ANN)*100, lw=1, color="#c0392b")
ax[2].set_ylabel("20d vol %")
plt.tight_layout(); plt.show()

**What is forecastable and what not?** Below we plot correlograms of returns and absolute returns. For a given timesseries, the k-th bar in a correlogram shows the correlation between a datapoint and its k-step lagged datapoint. For example, the first bar on the left panel shows the correlation between yesterdays returns and today returns. **<font color='greeb'>Run</font>** the cell below.

In [ ]:
def acf(x, nlags=30):
    x = pd.Series(x).dropna().values
    x = x - x.mean()
    return np.array([1.0] + [np.corrcoef(x[:-l], x[l:])[0, 1] for l in range(1, nlags+1)])

lags = np.arange(31)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
ax[0].bar(lags, acf(df.ret),       color="#999");    ax[0].set_title("ACF of returns")
ax[1].bar(lags, acf(df.ret.abs()), color="#c0392b"); ax[1].set_title("ACF of |returns|")
for a in ax: a.axhline(0, color="k", lw=.6); a.set_xlabel("lag (days)")
plt.tight_layout(); plt.show()

**<font color='yellow'>Discuss:</font>** What do the two correlograms tell you about the forecastability of returns? Do past returns provide information about the direction of future returns? What about the 'magnitude' of future returns? What does that tell you about the forecastability of volatility?


---
## 2 · The base model: where is vol *now*?

In this section we will start building a base model for forecasting the 20-day volatility. The model will be built out of two pieces:
- a **long-run average** — stable, but very slow to notice anything
- a **recent estimate** — More reactive, but also more noisy

This model is suitable as it captures two key properties of volatility: it clusters and it mean reverts.

**<font color='yellow'> Discuss: </font>** Why can the model above capture these two dynamical properties of vol?

The next cell loads all the machinery necessary for building the base model. No need to change anything, just **<font color='greeb'>run</font>** it.


In [ ]:
# ---- the machinery. Read it, don't edit it. -------------------------
def ann_vol(daily_var):
    return np.sqrt(daily_var * ANN) * 100

def var_series(d):
    """Daily variance. Days carrying a label in CONFIG['exclude_events'] are
    left out (section 4c)."""
    v = d[SETTLE_COL].values.copy()
    excl = CONFIG["exclude_events"]
    if not isinstance(excl, (list, tuple, set)):
        raise TypeError('CONFIG["exclude_events"] is a list of labels, e.g. '
                        '["earnings"]. Use [] to leave nothing out.')
    if excl:
        v[d["event_type"].isin(list(excl)).values] = np.nan
    return v

def long_run_var(d, i):
    """Average variance over CONFIG['lookback'] days up to i (None = all history)."""
    v = var_series(d)
    n = CONFIG["lookback"]
    seg = v[: i+1] if n is None else v[max(0, i-n+1): i+1]
    return float(np.nanmean(seg)) if np.isfinite(seg).any() else np.nan

def ewma_var(d, i, lam=None, lookback=250):
    """Exponentially weighted variance: recent days count for more."""
    lam = CONFIG["lam"] if lam is None else lam
    v = var_series(d)[max(0, i-lookback+1): i+1]
    v = v[~np.isnan(v)]
    if len(v) == 0: return np.nan
    w = lam ** np.arange(len(v)-1, -1, -1)
    return float(np.sum(w*v) / np.sum(w))

def recent_move(d, i, n=None):
    """The last n days' move, in sigmas, sign flipped so it is POSITIVE after a
    fall and NEGATIVE after a rally. Clipped to keep one wild day from dominating."""
    n = CONFIG["lev_n"] if n is None else n
    r = np.nansum(np.log(d["close"] / d["close"].shift(1)).values[max(0, i-n+1): i+1])
    s = np.sqrt(ewma_var(d, i, 0.94) * n)
    return float(np.clip(-r / s, -3, 3)) if s > 0 else 0.0

def count_upcoming(d, i, horizon, etype="earnings"):
    """Scheduled events inside the swap window. A desk knows these in advance."""
    return int((d["event_type"].iloc[i+1 : i+1+horizon] == etype).sum())

# ---- THE forecast. One function. You never edit it. -----------------
def forecast(d, i, horizon=None):
    """Annualised vol in percent for the next `horizon` days, using data up to i."""
    horizon = CONFIG["horizon"] if horizon is None else horizon

    # section 2 - the base level
    w = CONFIG["w"]
    var = (1 - w) * long_run_var(d, i)
    if w > 0:
        var += w * ewma_var(d, i)

    # section 4 - scheduled events inside the window, each at its own multiplier
    bump = sum((m - 1.0) * count_upcoming(d, i, horizon, t)
               for t, m in CONFIG["event_mult"].items())
    var *= 1 + bump / horizon

    # section 3 - direction of the recent move
    var *= np.exp(CONFIG["lev_g"] * recent_move(d, i))

    return ann_vol(var)

print("machinery ready. CONFIG currently says:")
print(f"   base   = long-run average over {CONFIG['lookback'] or 'all'} days")
print(f"   w      = {CONFIG['w']}   (0 = recent vol ignored)")
print(f"   lev_g  = {CONFIG['lev_g']}   event mult = {CONFIG['event_mult']['earnings']}")

### 2a — How much history is "the long-run average"?

Let's start with the first piece of the base model: the long-run average volatility. In order to measure it, we need to pick a window. Longer windows capture more histofrical data and are also more stable, but they are less sensitive to recent realization. **<font color='greeb'>Run</font>** the cell below to compute the long-run estimate over several windows and see how 'stable' such an average is.

In [ ]:
print("The long-run estimate at the end of the training period:")
print()
for n in [None, 1000, 500, 250, 120, 60]:
    CONFIG["lookback"] = n
    label = "all history" if n is None else f"last {n}d"
    print(f"   {label:<14} {ann_vol(long_run_var(df, TEST_START-1)):5.1f}%")

print()
print("How much each choice wanders as you move through the sample:")
print()
for n in [None, 1000, 500, 250, 120, 60]:
    CONFIG["lookback"] = n
    series = [ann_vol(long_run_var(df, i)) for i in range(TEST_START-500, TEST_START, 20)]
    label = "all history" if n is None else f"last {n}d"
    print(f"   {label:<14} stdev {np.nanstd(series):5.2f} vol points")

A short lookback tracks the current regime but wanders; a long one is steady but may be anchored to conditions long gone. No free lunch —  pick the trade-off you believe in. **<font color='orange'>Edit</font>** the 'lookback' parameter in the CONFIG dictionary below.

In [ ]:
CONFIG["lookback"] = None    # <-- None | 1000 | 500 | 250 | 120 | 60
print("long-run average over:", CONFIG["lookback"] or "all history")

### 2b — A faster estimate

Next up is our 'fast estimate'. To measure recent realization we will use an exponentially weighted moving average (EWMA) rather than a fixed uniform window -- can you think of why (<font color='yellow'>discuss</font>)? Such an EWMA needs a decay factor `lam` which controls how fast it forgets: `lam = 0.99` has a memory of months, `lam = 0.80` of about a week.

The cell below plots the EWMA over all the history for two different decay parameters. **<font color='greeb'>Run</font>**  it.

In [ ]:
sub = range(250, TEST_START)
dates = df.date.iloc[list(sub)]

fig, ax = plt.subplots(figsize=(11, 3.6))
for lam, col in [(0.99, "#1f3a63"), (0.80, "#c0392b"), ]:
    ax.plot(dates, [ann_vol(ewma_var(df, i, lam)) for i in sub],
            lw=1, color=col, label=f"EWMA lam={lam}")
ax.axhline(ann_vol(long_run_var(df, TEST_START-1)), color="k", ls="--", lw=1,
           label="long-run average")
ax.set_ylabel("vol %"); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

The 'strong' decay of `lam = 0.80` is much more reactive to brief moments of high realization, but also a lot noisier. The challenge in the next subsection will be to find the ideal decay parameter for combining this with the long term average. **<font color='yellow'>Discuss:</font>** What are your initial expectations? Do you think we want a strong or weak decay factor for our 'fast estimate'? 

### 2c — Combining both estimates and finding the right weights

Your forecast is:

    base  =  w · EWMA(lam)  +  (1 - w) · long-run average

With `w = 0` the model relies fuly on the long-run average, with `w = 1` we only look at the short-term estimate. The right answer
is in between, and it depends on `lam` too — so search both together.

The function `quick_score` is given to you: it scores the forecasting function for a given set of parameters. **<font color='greeb'>Run</font>** the next cell to initialize it. 

In [ ]:
def quick_score(horizon=None, start=260, stop=None, skip_event_windows=False):
    """QLIKE of the CURRENT CONFIG over the training period. Lower is better.

    QLIKE punishes under-forecasting volatility much harder than
    over-forecasting - the right asymmetry for someone who must quote a
    two-sided market. It never looks past TEST_START."""
    horizon = CONFIG["horizon"] if horizon is None else horizon
    stop = TEST_START if stop is None else stop
    fs, rs = [], []
    for i in range(start, stop):
        if skip_event_windows and count_upcoming(df, i, horizon) > 0:
            continue
        f = forecast(df, i, horizon)
        r = ann_vol(np.nanmean(df[SETTLE_COL].values[i+1 : i+1+horizon]))
        if np.isfinite(f) and np.isfinite(r) and f > 0 and r > 0:
            fs.append(f); rs.append(r)
    fs, rs = np.array(fs)**2, np.array(rs)**2
    return float(np.mean(rs/fs - np.log(rs/fs) - 1))

CONFIG["w"] = 0.0
print(f"your current model (long-run average only):  QLIKE = {quick_score():.4f}")

Above you see the score of the model with w = 0. Next, we want to find the optmal values lam and w.  **<font color='orange'>Fill</font>** in the two lists and run. Start coarse — you can refine around whatever
wins. **The grid is yours to design.**

In [ ]:
lams = []            # <-- your grid. add values,
ws   = []            # <--

if lams == [] or ws == []:
    print("Grid is empty. Please provide a grid to search...")
else:
    results, best = [], None
    for lam in lams:
        for w in ws:
            CONFIG["lam"], CONFIG["w"] = lam, w
            s = quick_score()
            results.append((lam, w, s))
            if best is None or s < best[2]:
                best = (lam, w, s)

    print(f"best:  lam = {best[0]}   w = {best[1]}   QLIKE = {best[2]:.4f}")
    print()
    print(pd.DataFrame(results, columns=["lam", "w", "qlike"])
            .pivot(index="lam", columns="w", values="qlike").round(4).to_string())

**<font color='yellow'>Discuss:</font>** What is the best parameter combination? How does this change if you alter the long-term average vol window?

Once you are happy with your parametrization, **<font color='orange'>register</font>** your choices in the CONFIG below

In [ ]:
CONFIG["lam"] = 0.94     # <-- your decay parameter
CONFIG["w"] = 0          # <-- your weight

---
## 3 · Does direction say anything about future volatility?

Traders will tell you vol goes up when the price goes down. Test it — and then
test whether knowing it makes your forecast any better. 

### 3a — Testing.

For each day: how big was the move, and what happened to vol over the following 20 days compared with the 20 before? That is the question we will try to answer here. 

Before we can answer that though, we need to be clear on what qualifies as a 'big move'. For a low vol stock, like an insurance company, a 3% move on a given day is big, but a high vol stock like a semiconductor company would regard such a move as a quiet day. Hence, we need to look at a stocks return relative to their past volatility. We use the relative return `z` that is defined as todays return divided by the standard deviation of squared returns over the last n days. In stats language, one speaks of sigmas (one sigma = one standard deviation).

**<font color='greeb'>Run</font>** the cell below to find out.

In [ ]:
def leverage_table(d, n=20):
    r    = np.log(d["close"] / d["close"].shift(1))
    v    = r**2                             # daily variance
    past = v.rolling(n).mean().shift(1)     # mean variance BEFORE today - today excluded
    fwd  = v.rolling(n).mean().shift(-n)    # mean variance AFTER today
    y    = np.log(fwd / past)               # the ratio between future variance and past variance -- our target.
    z    = r / np.sqrt(past)                # the 'relative' move of today
    ok   = y.notna() & z.notna() & np.isfinite(y) & (d["event_type"] == "")
    return pd.DataFrame({"y": y[ok], "z": np.clip(z[ok], -5, 5)})

tab  = leverage_table(df, CONFIG["horizon"])
by_q = tab.groupby(pd.qcut(tab.z, 7, labels=False)).agg(
           mean_z=("z", "mean"), mean_y=("y", "mean"))

print("Change in vol over the next 20 days, by today's move:")
print()
for _, row in by_q.iterrows():
    bar = "#" * int(max(row.mean_y, 0) * 60)
    print(f"   {row.mean_z:+5.2f} sigma   ->  {row.mean_y:+.3f}  {bar}")

fig, ax = plt.subplots(figsize=(7, 3.2))
bins = np.linspace(-3, 3, 19); ctr = (bins[:-1] + bins[1:]) / 2
mu = [tab.y[(tab.z >= bins[i]) & (tab.z < bins[i+1])].mean() for i in range(len(bins)-1)]
ax.plot(ctr, mu, "o-", color="#c0392b")
ax.axhline(0, color="k", lw=.8); ax.axvline(0, color="grey", lw=.8, ls="--")
ax.set_xlabel("today's relative move z"); ax.set_ylabel("log(RV next / RV past)")
ax.set_title("both ends turn up - but are they the same height?")
plt.tight_layout(); plt.show()

We see two things: a parabola shape which tells us that big moves are followed by elevated volatility and a slight asymmetry suggesting that down moves raise the vol more than up moves. <font color='yellow'>Discuss:</font> Which of these two characteristics did we already address in the previous section and which one not?

### 3b — Regression

Let's quantify the shape we just observed, using a regression:

    y  =  a·z^2  +  b·z  +  c

<font color='greeb'>Run</font> the cell below to regress future vol changes on this quadratic relation with the returns.

In [ ]:
def ols_hac(X, Y, lag):
    """Least squares with Newey-West standard errors. Consecutive 20-day windows
    share 19 of their 20 days, so residuals are badly autocorrelated and plain
    OLS errors would make everything look significant."""
    beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    u    = Y - X @ beta
    XtXi = np.linalg.inv(X.T @ X)
    S    = (X * u[:, None]).T @ (X * u[:, None])
    for l in range(1, lag + 1):
        G = (X[l:] * u[l:, None]).T @ (X[:-l] * u[:-l, None])
        S += (1 - l / (lag + 1)) * (G + G.T)
    V = XtXi @ S @ XtXi
    return beta, np.sqrt(np.diag(V))

Z = tab.z.values; Y = tab.y.values
(a, b, c), se = ols_hac(np.column_stack([Z**2, Z, np.ones(len(Z))]), Y,
                        CONFIG["horizon"])
print("   y = a*z^2 + b*z + c")
print()
print(f"   a  (magnitude) =  {a:+.4f},   t = {a/se[0]:+5.2f}")
print(f"   b  (direction) =  {b:+.4f},   t = {b/se[1]:+5.2f}")
print(f"   c  (intercept) =  {c:+.4f},   t = {c/se[2]:+5.2f}")
print()
print(f"   Minimum at -b/2a = {-b/(2*a) :+.2f}")

<font color='yellow'>Discuss:</font> What do these numbers tell you? Can you relate this to the volatility smile seen in options markets?

The numbers on the right are t-values: here simply the estimate divided by its standard error.


### 3c — Does it make your forecast better?

With knowledge of this 'directional' effect, can we improve our forecasting?

The model we will use here is as follows. Our base model that we optimized 
in the previous section is multiplied by `exp(g · recent_move)`, where
`recent_move` is the last `lev_n` days' move in sigmas, **sign flipped**: it is
positive after a fall and negative after a rally. So `g > 0` raises your forecast
after a selloff and lowers it after a rally. `g = 0` switches the term off.

**<font color='orange'>Sweep it:</font>**

In [ ]:
lev_gs = []            # <-- your grid, add values.
lev_ns  = []           # <--

if lev_gs == [] or lev_ns == []:
    print("Grid is empty. Please provide a grid to search...")
else:
    results, best = [], None
    for lev_g in lev_gs:
        for lev_n in lev_ns:
            CONFIG["lev_g"], CONFIG["lev_n"] = lev_g, lev_n
            s = quick_score(skip_event_windows=True)
            results.append((lev_g, lev_n, s))
            if best is None or s < best[2]:
                best = (lev_g, lev_n, s)

    print(f"best:  lev_g = {best[0]}   lev_n = {best[1]}   QLIKE = {best[2]:.4f}")
    print()
    print(pd.DataFrame(results, columns=["lev_g", "lev_n", "qlike"])
            .pivot(index="lev_g", columns="lev_n", values="qlike").round(4).to_string())

Finally, **<font color='orange'>pick</font>** your set of optimal values:

In [ ]:
CONFIG["lev_g"] = 0    # <-- the strenght of your directional corection
CONFIG["lev_n"] = 10   # <-- your time window to compare todays move with

---
## 4 · Events

The `event_type` and `comment` columns tell you what was scheduled. Some of
those labels predict volatility. **Some do not.** In this section we look into 
how events affect our volatility forecast and how we should deal with them. **<font color='greeb'>Run </font>** the cell below to start.

In [ ]:
def flag_days(d, types):
    if isinstance(types, str): types = [types]
    return d["event_type"].isin(types)

def compare_vol(d, mask, label="flagged", n_boot=2000):
    """Variance on flagged days relative to ordinary days, with a 95% interval.

    Measured on the TRAINING period only: the held-out rows are not allowed to
    shape any number that goes into your model. The interval comes from
    resampling the flagged days - few days means a wide interval."""
    tr   = d.iloc[:TEST_START]
    m    = np.asarray(mask)[:TEST_START]
    ev   = tr[SETTLE_COL].values[m];  ev = ev[~np.isnan(ev)]
    base = tr[SETTLE_COL].values[(tr["event_type"] == "").values]
    base = base[~np.isnan(base)]
    ratio = ev.mean() / base.mean()
    rng   = np.random.default_rng(0)
    boots = np.array([rng.choice(ev, len(ev)).mean() for _ in range(n_boot)]) / base.mean()
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"   {label:<10} n={len(ev):>4}   vol {ann_vol(ev.mean()):5.1f}%"
          f"  vs ordinary {ann_vol(base.mean()):5.1f}%   variance ratio {ratio:5.2f}x"
          f"   95% interval [{lo:4.2f}, {hi:5.2f}]")
    return ratio

def event_decay(d, etype, lags=range(-3, 6)):
    d    = d.iloc[:TEST_START]                      # training period only
    idx  = np.where(d["event_type"] == etype)[0]
    base = d.loc[d["event_type"] == "", SETTLE_COL].mean()
    out  = {}
    for L in lags:
        j = idx + L
        j = j[(j >= 0) & (j < len(d))]
        out[L] = d[SETTLE_COL].iloc[j].mean() / base
    s = pd.Series(out)
    ax = s.plot(kind="bar", figsize=(7, 2.8),
                color=["#999" if k != 0 else "#c0392b" for k in s.index])
    ax.axhline(1, color="k", lw=.8, ls="--")
    ax.set_title(f"{etype}: variance ratio by lag"); ax.set_xlabel("days from event")
    plt.tight_layout(); plt.show()
    return s

def split_score(horizon=None, start=260, stop=None):
    """QLIKE of the current CONFIG over the training windows, in three groups:
    windows with no earnings, windows containing an earnings, and all of them.
    One pass through the data. Returns {group: (QLIKE, number of windows)}."""
    horizon = CONFIG["horizon"] if horizon is None else horizon
    stop = TEST_START if stop is None else stop
    fs, rs, ev = [], [], []
    for i in range(start, stop):
        f = forecast(df, i, horizon)
        r = ann_vol(np.nanmean(df[SETTLE_COL].values[i+1 : i+1+horizon]))
        if np.isfinite(f) and np.isfinite(r) and f > 0 and r > 0:
            fs.append(f); rs.append(r); ev.append(count_upcoming(df, i, horizon) > 0)
    fs, rs, ev = np.array(fs), np.array(rs), np.array(ev)
    q = lambda m: float(np.mean(rs[m]**2/fs[m]**2 - np.log(rs[m]**2/fs[m]**2) - 1))
    return {"quiet": (q(~ev), int((~ev).sum())),
            "event": (q(ev),  int(ev.sum())),
            "all":   (q(np.ones(len(fs), bool)), len(fs))}

print("helpers ready")

### 4a — Which labels actually predict?

A ratio near **1.0** means the label tells you nothing.

Each ratio is measured on the training period only, and comes with a 95%
interval obtained by resampling the event days.

In [ ]:
print("variance on event days vs ordinary days")
print()
for t in ["earnings", "agm", "news"]:
    compare_vol(df, flag_days(df, t), label=t)

**<font color='yellow'>Discuss:</font>** Which of these events seem relevant and which ones do not?

### 4b — How big, and how long does it last?

Is an event like earnings an isolated event or something that affects also the volatility in the (non-event) days preceding and following it? **<font color='greeb'>Run</font>** the cell below to find out.

In [ ]:
_ = event_decay(df, "earnings")

The bars situated at a positive `days from event` are days that **follow** the event -- those with a negative `days from event` are days **preceding** the event. **<font color='yellow'>Discuss:</font>** Is the bar at lag **+1** elevated, or back to normal? 

That is the difference between pricing a *bump* and pricing a *regime*. **<font color='greeb'>Try</font>**  also `event_decay(df, "news")` or `event_decay(df, "agm")` below to compare.

In [ ]:
_ = event_decay(df, "news")
_ = event_decay(df, "agm")

Now **<font color='orange'>set</font>** the multipliers. `1.0` means "this label does nothing".

In [ ]:
CONFIG["event_mult"] = {
    "earnings": 1.0,     # <-- what you measured in 4a
    "agm":      1.0,
    "news":     1.0,
}
print(CONFIG["event_mult"])
print(f"QLIKE = {quick_score():.4f}")

n_with = sum(count_upcoming(df, i, CONFIG["horizon"]) > 0
             for i in range(260, TEST_START))
print(f"...of {TEST_START-260} training windows; of those {n_with} contain an earnings"
      f" ({n_with/(TEST_START-260)*100:.0f}%)")

# the same training windows scored twice: your multipliers off, then on
mine = dict(CONFIG["event_mult"])
CONFIG["event_mult"] = {t: 1.0 for t in mine}
off = split_score()
CONFIG["event_mult"] = mine
on  = split_score()
print()
print(f"   {'training windows':<26}{'n':>6}{'multipliers off':>18}{'your multipliers':>19}")
for lab, key in [("with no earnings", "quiet"), ("containing an earnings", "event"),
                 ("all", "all")]:
    print(f"   {lab:<26}{on[key][1]:>6}{off[key][0]:>18.4f}{on[key][0]:>19.4f}")

**<font color='yellow'>Discuss:</font>** Compare the three rows. In which windows can the multipliers change the score at all, and in which can they not? What does that mean for how much the single number on the last row can tell you?

### 4c — Which event days belong in the base level?

Your base level — the long-run average and the EWMA — is computed over **every**
day in the sample, earnings days included. Some of those days are several times
more volatile than normal, so your "ordinary level" is already inflated by them.

`exclude_events` is a **list of labels** whose days are left out when the base
level is measured. Each label is its own decision:

- `[]` — nothing left out; the base includes every event day (what you have now)
- `["earnings"]` — earnings days left out, every other day stays in
- `["earnings", "agm", "news"]` — every labelled day left out; the base is
  estimated on ordinary days only

Reason it through first, then **<font color='orange'>test</font>** any combination you want below.

In [ ]:
candidates = [                      # <-- add your own lists
    [],
    ["earnings", "agm", "news"],
]

print(f"   {'left out of the base':<34}{'base level':>11}{'no earnings':>13}{'earnings':>10}{'all':>9}")
for excl in candidates:
    CONFIG["exclude_events"] = excl
    s = split_score()
    base = ann_vol(long_run_var(df, TEST_START-1))
    print(f"   {str(excl):<34}{base:>10.1f}%{s['quiet'][0]:>13.4f}"
          f"{s['event'][0]:>10.4f}{s['all'][0]:>9.4f}")


Make you final choice for which events to exclude or not, by **<font color='orange'>editing</font>** the file below.

In [ ]:
CONFIG["exclude_events"] = []   # <-- your choice: a list of labels

print()
for t in ["earnings", "agm", "news"]:
    left_out = "yes" if t in CONFIG["exclude_events"] else "none"
    print(f"   {t:<9} left out of the base: {left_out:<4}  multiplier: {CONFIG['event_mult'][t]}")

---
## 5 · Final backtest  — **RUN**  ⏱ ~8 min

Everything so far used the training period. Now the honest test: the last 400
days, which nothing you tuned has seen.

Go ahead and **<font color='greeb'>run</font>**.

In [ ]:
def backtest(fn=None, horizon=None):
    horizon = CONFIG["horizon"] if horizon is None else horizon
    fn = forecast if fn is None else fn
    rows = []
    for i in range(TEST_START, len(df) - horizon - 1):
        f = fn(df, i, horizon)
        r = ann_vol(df[SETTLE_COL].iloc[i+1 : i+1+horizon].mean())
        if np.isfinite(f) and np.isfinite(r) and f > 0 and r > 0:
            rows.append((df.date.iloc[i], f, r, count_upcoming(df, i, horizon) > 0))
    out = pd.DataFrame(rows, columns=["date", "forecast", "realised", "event_window"])
    rs, fs = out.realised**2, out.forecast**2
    out.attrs["qlike"] = float(np.mean(rs/fs - np.log(rs/fs) - 1))
    out.attrs["bias"]  = float((out.forecast - out.realised).mean())
    out.attrs["mae"]   = float((out.forecast - out.realised).abs().mean())
    return out

naive     = lambda d, i, h: ann_vol(d[SETTLE_COL].iloc[max(0, i-h+1): i+1].mean())
rolling60 = lambda d, i, h: ann_vol(d[SETTLE_COL].iloc[max(0, i-59): i+1].mean())
hindsight = lambda d, i, h: ann_vol(d[SETTLE_COL].iloc[TEST_START:].mean())

res = {"naive (last 20d)": backtest(naive),
       "rolling 60d":      backtest(rolling60),
       "hindsight const":  backtest(hindsight),
       "MY FORECAST":      backtest()}

print(pd.DataFrame({
    "QLIKE":          {k: v.attrs["qlike"] for k, v in res.items()},
    "MAE (vol pts)":  {k: v.attrs["mae"]   for k, v in res.items()},
    "bias (vol pts)": {k: v.attrs["bias"]  for k, v in res.items()},
}).sort_values("QLIKE").round(3).to_string())

---
## 6 · Your parameter card  — **RUN**

**The final overview of your choices**: This is your model and what you will use for forecasting. (Go ahead and **<font color='greeb'>run</font>**)

In [ ]:
q = backtest().attrs
print(f"""
+----------------------------------------------+
|            VOL DESK - PARAMETER CARD         |
+----------------------------------------------+
|  team: ____________________                  |
|                                              |
|  long-run lookback  {str(CONFIG['lookback'] or 'all'):<25}|
|  EWMA lam           {CONFIG['lam']:<25}|
|  weight on recent   {CONFIG['w']:<25}|
|  direction g        {CONFIG['lev_g']:<25}|
|  direction lookback {CONFIG['lev_n']:<25}|
|  left out of base   {(','.join(CONFIG['exclude_events']) or 'none'):<25}|
|                                              |
|  event multipliers                           |
|    earnings         {CONFIG['event_mult']['earnings']:<25}|
|    agm              {CONFIG['event_mult']['agm']:<25}|
|    news             {CONFIG['event_mult']['news']:<25}|
|                                              |
|  out-of-sample QLIKE  {q['qlike']:<23.4f}|
|  out-of-sample MAE    {q['mae']:<23.2f}|
+----------------------------------------------+
""")

If you want to change anything, here's your last change: <font color='orange'>uncomment</font> the line below whose value you wish to modify. Then run the cell above to verify your changes.

In [ ]:
OLD_CONFIG = CONFIG.copy()

# --------- if you change your mind, use this cell to modify ------------

# CONFIG['lookback'] = ...
# CONFIG['lam'] = ...
# CONFIG['w'] = ...
# CONFIG['lev_g'] = ...
# CONFIG['lev_n'] = ...
# CONFIG['exclude_events'] = ...
# CONFIG['event_mult']['earnings'] = ...
# CONFIG['event_mult']['agm'] = ...
# CONFIG['event_mult']['news'] = ...

for key in CONFIG.keys():
    if CONFIG[key] != OLD_CONFIG[key]:
        print(f"Changed {key} from {OLD_CONFIG[key]} to {CONFIG[key]}")


---
## 7 · Game time  — **RUN ONE CELL PER ROUND**

The game continues the same stock. Your 2000 rows stop the day before round 1 —
there is no second dataset and no new company.

Run the cell below to load all the data. Everything is preset, so all fine to just **<font color='greeb'>run</font>**.

In [ ]:
ROUND_URL = "https://raw.githubusercontent.com/avspaendonck/quant/refs/heads/main/round_{:02d}.csv"     # <-- instructor pastes the pattern, e.g.
CAL_URL   = "https://raw.githubusercontent.com/avspaendonck/quant/refs/heads/main/calendar.csv"     # <-- and the calendar, e.g. ".../rounds/calendar.csv"

COLS = ["date", "open", "high", "low", "close", "event_type", "comment"]

def history_at(r):
    """Everything you know at the start of round r: all prices revealed so far,
    then the forward calendar with no prices attached."""
    parts = [df[COLS]]
    for k in range(1, r):                       # windows already settled
        src = (ROUND_URL or "rounds/round_{:02d}.csv").format(k)
        parts.append(pd.read_csv(src, parse_dates=["date"])[COLS])
    hist = pd.concat(parts, ignore_index=True)

    cal = pd.read_csv(CAL_URL or "rounds/calendar.csv", parse_dates=["date"])
    cal = cal.iloc[(r-1) * CONFIG["horizon"]:].copy()    # this window onward
    for c in ["open", "high", "low", "close"]:
        cal[c] = np.nan

    d = pd.concat([hist, cal[COLS]], ignore_index=True)
    d["event_type"] = d["event_type"].fillna("")
    d["comment"]    = d["comment"].fillna("")
    d["ret"]        = np.log(d["close"]).diff()
    return add_estimators(d)

def quote(r, horizon=None):
    horizon = CONFIG["horizon"] if horizon is None else horizon
    d = history_at(r)
    i = int(d["close"].last_valid_index())
    fair = forecast(d, i, horizon)

    rv = lambda n: ann_vol(d[SETTLE_COL].iloc[i-n+1 : i+1].mean())
    print(f"  ROUND {r}             {horizon}d vol swap as of {d.date.iloc[i]:%Y-%m-%d}"
            f"   ({i+1} days of history)")
    print("")
    print(f"  long-run anchor    {ann_vol(long_run_var(d, i)):5.1f}%")
    print(f"  short-term ewma    {ann_vol(ewma_var(d, i, )):5.1f}%")
    print(f"  directional corr.   {np.exp(CONFIG['lev_g'] * recent_move(d, i)):.2f}")
    print(f"  Recent move         {recent_move(d, i):+.2f} sigma  (+ = has fallen)")
    print("")
    cal = d.iloc[i+1 : i+1+horizon]
    cal = cal[cal.event_type != ""]
    if len(cal):
        print("  Upcoming events:")
        for _, e in cal.iterrows():
            print(f"     {e.date:%Y-%m-%d}  {e.event_type:<9} {e.comment}")
    else:
        print("  Upcoming events:    nothing scheduled")
    print("")
    print(f"  YOUR FAIR          {fair:5.1f}%")
    return fair

**<font color='greeb'>Run</font>** the cell below to forecast the first and subsequent rounds.

In [ ]:
ROUND = 1          # <-- set the round here
_ = quote(ROUND)